# Approach 2 — pretrained vision encoder + GPT-2 decoder

Fine-tune a Hugging Face VisionEncoderDecoder checkpoint on frontal chest X-rays and report text. This is an alternative baseline to the custom attention LSTM and Qwen3-VL notebook. It is research code only, not a clinical system.

## Runtime and data controls

A GPU is strongly recommended. The first run downloads the pretrained `nlpconnect/vit-gpt2-image-captioning` checkpoint. Start with 512 training studies and 64 validation studies; change either limit to `None` only after a successful end-to-end run. Do not use test results to select checkpoints.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cxr_second_reader project folder in VS Code/Jupyter.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Local project folder:", REPO_ROOT)


In [ ]:
import json
from functools import partial
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoTokenizer, VisionEncoderDecoderModel
from src.experiment_utils import study_rows, ImageTextRows
from src.evaluate import evaluate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "nlpconnect/vit-gpt2-image-captioning"
BATCH_SIZE, EPOCHS, MAX_LENGTH = 4, 1, 128
TRAIN_LIMIT, VAL_LIMIT = 512, 64
print("device:", DEVICE)

In [ ]:
train_df = study_rows("train", TRAIN_LIMIT)
val_df = study_rows("validation", VAL_LIMIT)
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def collate_caption_rows(rows):
    pixels = image_processor(images=[r["image"] for r in rows], return_tensors="pt")["pixel_values"]
    target = tokenizer(
        [r["text"] for r in rows], padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )["input_ids"]
    target[target == tokenizer.pad_token_id] = -100
    return pixels, target

train_loader = DataLoader(ImageTextRows(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          pin_memory=(DEVICE.type == "cuda"), collate_fn=collate_caption_rows)
val_loader = DataLoader(ImageTextRows(val_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        pin_memory=(DEVICE.type == "cuda"), collate_fn=collate_caption_rows)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id or tokenizer.eos_token_id
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
print("train/validation:", len(train_df), len(val_df))

In [ ]:
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for pixels, labels in train_loader:
        pixels = pixels.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = model(pixel_values=pixels, labels=labels).loss
        if not torch.isfinite(loss): raise RuntimeError("Non-finite loss; stop and inspect the runtime.")
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); losses.append(float(loss.detach().cpu()))
    print(f"epoch {epoch+1}/{EPOCHS} mean_loss={np.mean(losses):.4f}")

@torch.inference_mode()
def generate_on_loader(loader, limit=24):
    model.eval(); preds=[]; refs=[]
    for pixels, labels in loader:
        generated = model.generate(
            pixel_values=pixels.to(DEVICE), max_new_tokens=MAX_LENGTH,
            num_beams=2, do_sample=False,
        )
        preds.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        clean = labels.masked_fill(labels == -100, tokenizer.pad_token_id)
        refs.extend(tokenizer.batch_decode(clean, skip_special_tokens=True))
        if len(preds) >= limit: break
    return [x.strip() for x in preds[:limit]], [x.strip() for x in refs[:limit]]

val_predictions, val_references = generate_on_loader(val_loader, limit=24)
val_metrics = evaluate(val_predictions, val_references)
print("validation text-overlap metrics (not clinical accuracy):", val_metrics)
OUT = Path("outputs/03_vision_encoder_decoder"); OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT / "model"); image_processor.save_pretrained(OUT / "model"); tokenizer.save_pretrained(OUT / "model")
(OUT / "validation_metrics.json").write_text(json.dumps(val_metrics, indent=2), encoding="utf-8")
print("saved:", OUT.resolve())

## Final test evaluation

After choosing settings with validation only, load the saved model and evaluate `study_rows("test", None)` once. Save predictions with `uid` and the generated/reference reports so errors can be reviewed. BLEU and ROUGE-L indicate wording overlap only.

## Final held-out test evaluation
Run once after all model and threshold choices are frozen on validation. This cell writes per-study predictions and metrics to the model output folder.


In [ ]:
# Run once after settings are fixed using validation. Saves per-study predictions.
import pandas as pd
from src.experiment_utils import study_rows, ImageTextRows
from torch.utils.data import DataLoader

test_df = study_rows("test", None)
test_loader = DataLoader(
    ImageTextRows(test_df), batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
    collate_fn=collate_caption_rows,
)
model.eval(); predictions=[]; references=[]
with torch.inference_mode():
    for pixels, labels in test_loader:
        generated = model.generate(pixel_values=pixels.to(DEVICE), max_new_tokens=MAX_LENGTH, num_beams=2, do_sample=False)
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
        clean = labels.masked_fill(labels == -100, tokenizer.pad_token_id)
        references.extend(tokenizer.batch_decode(clean, skip_special_tokens=True))
predictions = [x.strip() for x in predictions]
references = [x.strip() for x in references]
metrics = evaluate(predictions, references)
rows = pd.DataFrame({"uid": test_df["uid"].astype(str), "prediction": predictions, "reference": references})
OUT.mkdir(parents=True, exist_ok=True)
rows.to_csv(OUT / "test_predictions.csv", index=False)
(OUT / "test_metrics.json").write_text(json.dumps({"n": len(rows), **metrics}, indent=2), encoding="utf-8")
print("Held-out test text-overlap metrics (not clinical accuracy):", metrics)
print("Saved", OUT / "test_predictions.csv")
